In [ ]:
import json
from evaluate import load
import torch

# Load evaluation metrics
rouge = load("rouge")
bertscore = load("bertscore")

# Load summary pairs from files
def parse_summary_file(filepath):
    """Extract summaries from formatted text file"""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    summaries = []
    sections = content.split('=' * 80)
    
    for section in sections:
        if not section.strip() or 'SUMMARIES' in section:
            continue
        
        lines = section.strip().split('\n')
        paper_name = None
        summary_text = []
        
        for i, line in enumerate(lines):
            if line.startswith('Paper: '):
                paper_name = line.replace('Paper: ', '').strip()
            elif line.startswith('-' * 80):
                summary_text = lines[i+1:]
                break
        
        if paper_name:
            summary = '\n'.join(summary_text).strip()
            summary = summary.split('\n' + '=' * 80)[0].strip()
            summaries.append({'paper': paper_name, 'summary': summary})
    
    return summaries

# Parse summary files
summaries_set1 = parse_summary_file('new_paper_summaries_set1.txt')
summaries_set2 = parse_summary_file('new_paper_summaries_set2.txt')

# Score summaries with reward model
def get_reward_score(text):
    """Get reward score for a summary text"""
    inputs = reward_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = reward_model(**inputs)
        score = outputs.logits[0][0].item()
    return score

# Score all summaries and determine preferences
preferred_summaries = []
rejected_summaries = []
reward_scores = []

for item1, item2 in zip(summaries_set1, summaries_set2):
    score1 = get_reward_score(item1['summary'])
    score2 = get_reward_score(item2['summary'])
    
    # Higher score is preferred
    if score1 > score2:
        preferred_summaries.append(item1['summary'])
        rejected_summaries.append(item2['summary'])
        reward_scores.append({
            'paper': item1['paper'],
            'preferred_set': 1,
            'preferred_score': score1,
            'rejected_score': score2,
            'score_diff': score1 - score2
        })
    else:
        preferred_summaries.append(item2['summary'])
        rejected_summaries.append(item1['summary'])
        reward_scores.append({
            'paper': item2['paper'],
            'preferred_set': 2,
            'preferred_score': score2,
            'rejected_score': score1,
            'score_diff': score2 - score1
        })

# Compute ROUGE metrics (preferred vs rejected)
rouge_results = rouge.compute(
    predictions=preferred_summaries, 
    references=rejected_summaries
)

# Compute BERTScore metrics (preferred vs rejected)
bertscore_results = bertscore.compute(
    predictions=preferred_summaries, 
    references=rejected_summaries, 
    lang="en"
)

# Aggregate results
results = {
    'num_papers': len(reward_scores),
    'set1_preferred': sum(1 for r in reward_scores if r['preferred_set'] == 1),
    'set2_preferred': sum(1 for r in reward_scores if r['preferred_set'] == 2),
    'avg_score_diff': sum(r['score_diff'] for r in reward_scores) / len(reward_scores),
    'rouge': {
        'rouge1': rouge_results['rouge1'],
        'rouge2': rouge_results['rouge2'],
        'rougeL': rouge_results['rougeL'],
        'rougeLsum': rouge_results['rougeLsum']
    },
    'bertscore': {
        'precision': sum(bertscore_results['precision']) / len(bertscore_results['precision']),
        'recall': sum(bertscore_results['recall']) / len(bertscore_results['recall']),
        'f1': sum(bertscore_results['f1']) / len(bertscore_results['f1'])
    }
}

# Save detailed results
with open("evaluation_results.json", "w") as f:
    json.dump({
        'summary_statistics': results,
        'individual_scores': reward_scores
    }, f, indent=2)

# Display results
print("Evaluation Results")
print("=" * 60)
print(f"Papers evaluated: {results['num_papers']}")
print(f"Set 1 preferred: {results['set1_preferred']}")
print(f"Set 2 preferred: {results['set2_preferred']}")
print(f"Average score difference: {results['avg_score_diff']:.4f}")
print("\nROUGE Scores:")
print(f"  ROUGE-1: {results['rouge']['rouge1']:.4f}")
print(f"  ROUGE-2: {results['rouge']['rouge2']:.4f}")
print(f"  ROUGE-L: {results['rouge']['rougeL']:.4f}")
print("\nBERTScore:")
print(f"  Precision: {results['bertscore']['precision']:.4f}")
print(f"  Recall: {results['bertscore']['recall']:.4f}")
print(f"  F1: {results['bertscore']['f1']:.4f}")
print("\nResults saved to: evaluation_results.json")